## Imports & Load

In [11]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/mercari_sample.csv")
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loaded: 50,000 rows × 8 columns


## Pre-Cleaning Audit

One final null check before touching anything — this is the baseline we compare against after cleaning.

In [12]:
# Null rate per column — drives every fill-vs-drop decision below
(
    df.isnull().sum()
      .to_frame(name="null_count")
      .assign(null_pct=lambda x: (x["null_count"] / len(df) * 100).round(2))
      .query("null_count > 0")
)

,null_count,null_pct
category_name,237,0.47
brand_name,21584,43.17


In [13]:
# Zero-price rows — free items cannot be log-transformed and distort price models
print(f"Zero-price rows: {(df['price'] == 0).sum()}")

Zero-price rows: 40


## Cleaning

In [14]:
# brand_name — 43.2% null: filling is the only option that preserves row count
# "No Brand" is a legitimate business category, not a placeholder;
# unbranded items show distinct pricing patterns worth modelling separately
df["brand_name"] = df["brand_name"].fillna("No Brand")
df["brand_name"] = df["brand_name"].str.lower().str.strip()

In [15]:
# category_name — 0.47% null (237 rows): below the 5% drop threshold
# dropping avoids a silent NaN bucket in every future groupby on category
df = df.dropna(subset=["category_name"])
print(f"After dropping null categories: {df.shape[0]:,} rows")

After dropping null categories: 49,763 rows


In [16]:
# price == 0 — free items are not pricing anomalies to fix; they are a
# different product type entirely and would anchor any price model toward zero
df = df[df["price"] > 0]
print(f"After removing zero-price rows: {df.shape[0]:,} rows")

After removing zero-price rows: 49,725 rows


In [17]:
# Duplicates — confirmed 0 in notebook 01, but we drop defensively
# in case the source file ever changes
before = len(df)
df = df.drop_duplicates()
print(f"Duplicates removed: {before - len(df)}")

Duplicates removed: 0


## Post-Cleaning Verification

Confirm every issue from the audit has been resolved — no nulls, no zero-price rows.

In [18]:
# All null counts should be zero
assert df.isnull().sum().sum() == 0, "Nulls remain — check cleaning steps above"

# All prices should be positive
assert (df["price"] > 0).all(), "Zero or negative prices remain"

print("✓ All checks passed")
print(f"  Clean shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Rows removed: {50_000 - df.shape[0]:,} ({(50_000 - df.shape[0]) / 50_000 * 100:.2f}% of original)")

✓ All checks passed
  Clean shape : 49,725 rows × 8 columns
  Rows removed: 275 (0.55% of original)


In [19]:
# Final null audit — every column should show 0
df.isnull().sum()

train_id             0
name                 0
item_condition_id    0
category_name        0
brand_name           0
price                0
shipping             0
item_description     0
dtype: int64